# 02. Моделирование, калибровка, оптимизация матрицы затрат и SHAP

Ноутбук демонстрирует валидацию ансамбля CatBoost на 5 стратифицированных фолдах, оценку надежности калибровки вероятностей, оптимизацию порога классификации по финансовой матрице потерь HR и итоговую оценку на изолированном тесте с объяснением факторов риска через SHAP.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.config import DATA_PROCESSED_DIR, TARGET_COL
from src.evaluate import (
    calculate_ensemble_predictions,
    load_trained_ensemble,
    prepare_holdout_test_data,
)
from src.metrics import evaluate_threshold_metrics, find_optimal_threshold

## 1. Загрузка OOF-предсказаний и оптимизация рабочего порога

In [ ]:
df_oof = pd.read_parquet(DATA_PROCESSED_DIR / "oof_predictions.parquet")
y_oof = df_oof[TARGET_COL].values
cal_oof = df_oof["calibrated_probability"].values

opt_threshold, df_costs = find_optimal_threshold(y_oof, cal_oof)
print(f"Оптимальный рабочий порог по матрице затрат: tau* = {opt_threshold:.2f}")

plt.figure(figsize=(8, 4))
plt.plot(df_costs["threshold"], df_costs["total_cost"] / 1000.0, color="forestgreen", lw=2)
plt.axvline(opt_threshold, color="crimson", linestyle="--", label=f"tau* = {opt_threshold:.2f}")
plt.title("Кривая совокупных потерь компании от выбора порога")
plt.xlabel("Порог классификации (tau)")
plt.ylabel("Суммарные затраты, тыс. $")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 2. Финальная оценка на изолированном тесте (Hold-out Test Set, N=206)

In [ ]:
models, calibrator, transformer = load_trained_ensemble()
X_test_trans, y_test, test_pool = prepare_holdout_test_data(transformer)
raw_probs, cal_test_probs = calculate_ensemble_predictions(models, test_pool, calibrator)

test_metrics = evaluate_threshold_metrics(
    y_true=y_test, y_prob=cal_test_probs, threshold=opt_threshold
)
print("--- Результаты на изолированном тесте ---")
print(f"Test PR-AUC:   {test_metrics['pr_auc']:.4f}")
print(f"Test ROC-AUC:  {test_metrics['roc_auc']:.4f}")
print(f"Test Recall:   {test_metrics['recall']:.2%}")
print(f"Test Precision:{test_metrics['precision']:.2%}")
print(f"Предотвращенный ущерб: {test_metrics['cost_reduction']:,.2f} $")

## 3. Интерпретация влияния признаков через SHAP Beeswarm

In [ ]:
import shap

shap_list = [m.get_feature_importance(test_pool, type="ShapValues")[:, :-1] for m in models]
ensemble_shap = np.mean(shap_list, axis=0)

plt.figure(figsize=(10, 6))
shap.summary_plot(ensemble_shap, X_test_trans, show=False)
plt.title("SHAP Beeswarm: Влияние факторов на вероятность увольнения")
plt.tight_layout()
plt.show()